# Retrieval + Generation Pipeline

This notebook connects ChromaDB legal corpus to a powerful LLM via Groq API.

**What this does:**
```
User Question
     ↓
Search ChromaDB (Retrieval)
     ↓
Format Retrieved Chunks as Numbered Sources
     ↓
Send to Llama 3.3 70B via Groq (Generation)
     ↓
Cited Legal Answer with Source References
```

In [1]:
!pip install -q groq

In [2]:
import os
from pathlib import Path

os.environ["GROQ_API_KEY"] = ""

PROJECT_ROOT   = Path("/Users/ritesh/Downloads/IP_Shakti_Sahayak")
CHROMA_DB_PATH = PROJECT_ROOT / "chromadb_store"


EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"   # Same model used in ingestion
COLLECTION_NAME = "ip_sakti_legal_corpus"     

# Groq LLM model — Llama 3.3 70B is the best free model for legal reasoning
LLM_MODEL = "openai/gpt-oss-120b"

TOP_K = 7

print(f"📂 ChromaDB  : {CHROMA_DB_PATH}")
print(f"🧠 Embedding : {EMBEDDING_MODEL}")
print(f"🤖 LLM       : {LLM_MODEL}")
print(f"🔍 Top-K     : {TOP_K} chunks per query")
print(f"🔑 API Key   : {'✅ Set' if os.environ.get('GROQ_API_KEY', '').startswith('gsk_') else '❌ NOT SET — paste your key above!'}")

📂 ChromaDB  : /Users/ritesh/Downloads/IP_Shakti_Sahayak/chromadb_store
🧠 Embedding : BAAI/bge-small-en-v1.5
🤖 LLM       : openai/gpt-oss-120b
🔍 Top-K     : 7 chunks per query
🔑 API Key   : ✅ Set


In [3]:
import chromadb

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))
collection = chroma_client.get_collection(name=COLLECTION_NAME)

print(f"✅ ChromaDB loaded successfully!")
print(f"   Total chunks in database: {collection.count()}")

✅ ChromaDB loaded successfully!
   Total chunks in database: 49689


## Load Embedding Model

Same model as Phase 1 (already cached on disk, loads fast).

In [4]:
from sentence_transformers import SentenceTransformer
import time

print(f"⏳ Loading embedding model: {EMBEDDING_MODEL}")
start = time.time()
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device="mps")
elapsed = time.time() - start

print(f"✅ Model loaded in {elapsed:.1f}s (cached from Phase 1)")
print(f"   Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

⏳ Loading embedding model: BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Model loaded in 8.9s (cached from Phase 1)
   Embedding dimension: 384


/var/folders/g8/myg8rccn3jvgj765h6v_859c0000gn/T/ipykernel_11582/2519351904.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"   Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


## Initialize Groq Client

Connects to Groq's ultra-fast inference API.

In [5]:
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Quick test — make sure the API key works
try:
    test_response = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "Say 'API connected' in exactly 2 words."}],
        max_tokens=10,
        temperature=0,
    )
    print(f"✅ Groq API connected!")
    print(f"   Model: {LLM_MODEL}")
    print(f"   Test response: {test_response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Groq API error: {e}")
    print(f"   Check your API key in Cell 2!")

✅ Groq API connected!
   Model: openai/gpt-oss-120b
   Test response: 


## The Retrieval Function

Searches ChromaDB and returns the most relevant legal chunks with metadata.  
Supports optional filtering by `jurisdiction` and `domain`.

In [6]:
from typing import List, Dict, Optional, Tuple


def retrieve(query: str,
             n_results: int = TOP_K,
             jurisdiction: Optional[str] = None,
             domain: Optional[str] = None) -> List[Dict]:
    """
    Search the legal corpus and return ranked results with metadata.
    
    Args:
        query: User's question
        n_results: Number of chunks to retrieve
        jurisdiction: 'india' or 'international' (optional filter)
        domain: 'patent', 'biodiversity', 'drug_regulation', etc. (optional)
    
    Returns:
        List of dicts, each with 'content', 'metadata', 'similarity'
    """
    # Build metadata filter
    where = None
    conditions = []
    if jurisdiction:
        conditions.append({"jurisdiction": {"$eq": jurisdiction}})
    if domain:
        conditions.append({"domain": {"$eq": domain}})
    if len(conditions) == 1:
        where = conditions[0]
    elif len(conditions) > 1:
        where = {"$and": conditions}

    # Embed the query
    query_embedding = embedding_model.encode(
        query, normalize_embeddings=True
    ).tolist()

    # Search ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=where,
        include=["documents", "metadatas", "distances"],
    )

    # Format results
    formatted = []
    if results["documents"] and results["documents"][0]:
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ):
            formatted.append({
                "content": doc,
                "metadata": meta,
                "similarity": round(1 - dist, 3),
            })

    return formatted


print("✅ Retrieval function ready")

✅ Retrieval function ready


## The System Prompt (The Brain of IP-SAKTI)

This is the most critical prompt engineering for the entire project.  
It defines how the AI behaves, cites sources, and avoids hallucination.

In [7]:
SYSTEM_PROMPT = """You are **IP-SAKTI Sahayak**, an expert AI assistant for Intellectual Property (IP) 
and regulatory guidance in Ayurveda. You help practitioners, researchers, AYUSH startups, 
and IP professionals navigate India's complex IP and regulatory landscape for traditional 
medicine products.

## YOUR CRITICAL RULES:

### Rule 1: ONLY Use the Provided Sources
You MUST base your answer ONLY on the source documents provided below. 
Do NOT use your general training knowledge to state legal facts, section numbers, 
or regulatory requirements. If the sources don't contain enough information, 
clearly say: "Based on the available sources, I could not find sufficient information 
on this specific point. Please consult a qualified IP professional."

### Rule 2: ALWAYS Cite Your Sources
Every legal fact, section reference, or regulatory requirement you mention MUST 
have an inline citation like [Source 1], [Source 2], etc. referring to the numbered 
sources provided. If you cannot cite a source for a claim, do NOT make that claim.

### Rule 3: Structure Your Answer
Organize your response clearly with:
- A direct, concise answer to the question first
- Supporting legal analysis with citations
- Practical next steps or recommendations (if applicable)
- A "Sources Used" section at the end listing all cited sources

### Rule 4: Provide Guidance, Not Legal Advice
You provide informational guidance to help users understand the legal and regulatory 
landscape. You are NOT a lawyer. Always end with the disclaimer.

### Rule 5: Be Precise with Legal Language
When referencing Acts, use the exact name and section numbers from the sources. 
Never paraphrase section numbers or invent provisions that aren't in the sources.

## RESPONSE FORMAT:

### Answer
[Your direct answer with inline citations like [Source 1], [Source 3]]

### Legal Analysis
[Detailed analysis referencing specific sections, rules, or guidelines from the sources]

### Recommended Next Steps
[Practical actions the user should take]

### Sources Used
[List of all sources you cited, with their document name and section reference]

---
⚠️ **Disclaimer:** This is informational guidance only, not legal advice. 
Please consult a qualified IP attorney or regulatory professional for specific cases.
"""

print("✅ System prompt defined")
print(f"   Length: {len(SYSTEM_PROMPT)} characters")

✅ System prompt defined
   Length: 2268 characters


## The Generation Function

Takes retrieved chunks + user query → sends to Groq → returns a cited answer.

In [8]:
def format_sources_for_llm(retrieved_chunks: List[Dict]) -> str:
    """
    Format retrieved chunks into a numbered source list for the LLM.
    
    Each source includes:
    - Source number
    - Document name
    - Authority
    - Section reference (if available)
    - The actual text content
    """
    sources_text = []

    for i, chunk in enumerate(retrieved_chunks, 1):
        meta = chunk["metadata"]
        source_file = meta.get("source_file", "Unknown")
        authority = meta.get("authority", "Unknown")
        jurisdiction = meta.get("jurisdiction", "Unknown")
        section = meta.get("section_reference", "")
        title = meta.get("title", "")
        similarity = chunk["similarity"]

        header = f"[Source {i}]"
        header += f"\n  Document: {source_file}"
        header += f"\n  Authority: {authority}"
        header += f"\n  Jurisdiction: {jurisdiction}"
        if section:
            header += f"\n  Section: {section}"
        if title:
            header += f"\n  Title: {title}"
        header += f"\n  Relevance: {similarity}"
        header += f"\n  Content:\n{chunk['content']}"

        sources_text.append(header)

    return "\n\n" + ("=" * 50) + "\n\n".join(sources_text)


def generate_answer(query: str,
                    retrieved_chunks: List[Dict],
                    temperature: float = 0.1) -> str:
    """
    Generate a cited legal answer using Groq.
    
    Args:
        query: User's question
        retrieved_chunks: List of chunks from retrieve()
        temperature: 0.0-1.0 (lower = more factual, higher = more creative)
    
    Returns:
        The LLM's cited answer as a string
    """
    if not retrieved_chunks:
        return "❌ No relevant documents found in the corpus for this query."

    # Format sources for the LLM
    sources_text = format_sources_for_llm(retrieved_chunks)

    # Build the user message
    user_message = f"""Based ONLY on the following authoritative sources, answer this question:

## QUESTION:
{query}

## SOURCES:
{sources_text}

Remember: Cite every legal fact with [Source X]. Do NOT make claims without a source."""

    # Call Groq API
    try:
        response = groq_client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_message},
            ],
            temperature=temperature,
            max_tokens=2048,
            top_p=0.9,
        )
        return response.choices[0].message.content

    except Exception as e:
        return f"❌ Groq API error: {e}"


print("✅ Generation function ready")

✅ Generation function ready


## The Complete RAG Pipeline

This is the **main function** that ties everything together:  
`Question → Retrieve → Generate → Display`

In [9]:
def ask_ip_sakti(query: str,
                 jurisdiction: Optional[str] = None,
                 domain: Optional[str] = None,
                 show_sources: bool = True) -> str:
    """
    The full IP-SAKTI RAG pipeline.
    
    Args:
        query: Your legal question
        jurisdiction: 'india' or 'international' (optional)
        domain: 'patent', 'biodiversity', 'drug_regulation',
                'traditional_knowledge', 'food_regulation',
                'trademark', 'copyright', 'international_ip' (optional)
        show_sources: Whether to display the retrieved source chunks
    
    Returns:
        The AI's cited answer
    """
    print(f"{'='*70}")
    print(f"🔍 QUESTION: {query}")
    if jurisdiction or domain:
        filters = []
        if jurisdiction: filters.append(f"jurisdiction={jurisdiction}")
        if domain: filters.append(f"domain={domain}")
        print(f"   Filters: {', '.join(filters)}")
    print(f"{'='*70}")

    # ── Step 1: Retrieve ──
    print(f"\n📚 Step 1: Searching legal corpus...")
    chunks = retrieve(query, n_results=TOP_K,
                      jurisdiction=jurisdiction, domain=domain)

    if not chunks:
        print("❌ No relevant documents found.")
        return "No relevant documents found."

    print(f"   Found {len(chunks)} relevant chunks:")
    for i, c in enumerate(chunks, 1):
        meta = c["metadata"]
        sec = meta.get("section_reference", "")
        sec_str = f" | {sec}" if sec else ""
        print(f"   [{i}] {meta.get('source_file', '?')}{sec_str} "
              f"(similarity: {c['similarity']})")

    # ── Step 2: Generate ──
    print(f"\n🤖 Step 2: Generating cited answer via {LLM_MODEL}...")
    start = time.time()
    answer = generate_answer(query, chunks)
    gen_time = time.time() - start
    print(f"   Generated in {gen_time:.1f} seconds")

    # ── Step 3: Display ──
    print(f"\n{'='*70}")
    print(f"🏛️  IP-SAKTI SAHAYAK — RESPONSE")
    print(f"{'='*70}")
    print(answer)
    print(f"{'='*70}\n")

    return answer


print("✅ Full RAG pipeline ready!")
print("   Use: ask_ip_sakti('your question here')")

✅ Full RAG pipeline ready!
   Use: ask_ip_sakti('your question here')


## Test 1: Patent + Traditional Knowledge

The classic question that every SIH judge will ask.

In [10]:
answer = ask_ip_sakti(
    query="Can I patent an Ayurvedic formulation that uses Ashwagandha and Brahmi "
          "in a novel nanoparticle delivery system?",
    jurisdiction="india",
    domain="patent",
)

🔍 QUESTION: Can I patent an Ayurvedic formulation that uses Ashwagandha and Brahmi in a novel nanoparticle delivery system?
   Filters: jurisdiction=india, domain=patent

📚 Step 1: Searching legal corpus...
   Found 7 relevant chunks:
   [1] Guidelines for Examination of Ayush Related Inventions-2025.pdf (similarity: 0.738)
   [2] Comments of Organisation of pharmaceuticals Producers of India, Mumbai dated April 10, 2008.pdf (similarity: 0.737)
   [3] Guidelines for Examination of Ayush Related Inventions-2025.pdf (similarity: 0.734)
   [4] Guidelines for Processing of Patent Applications relating to Traditional Knowledge and Biological Material - 2012.pdf (similarity: 0.724)
   [5] Guidelines for Examination of Ayush Related Inventions-2025.pdf (similarity: 0.722)
   [6] Guidelines for Examination of Ayush Related Inventions-2025.pdf (similarity: 0.721)
   [7] WO Ga 34, China.pdf (similarity: 0.721)

🤖 Step 2: Generating cited answer via openai/gpt-oss-120b...
   Generated in 4.4 seco

## Test 2: ABS Compliance

Can the system handle Access and Benefit Sharing questions?

In [11]:
answer = ask_ip_sakti(
    query="I am a foreign researcher who wants to study medicinal plants from Kerala. "
          "What ABS permissions do I need from the National Biodiversity Authority?",
    jurisdiction="india",
    domain="biodiversity",
)

🔍 QUESTION: I am a foreign researcher who wants to study medicinal plants from Kerala. What ABS permissions do I need from the National Biodiversity Authority?
   Filters: jurisdiction=india, domain=biodiversity

📚 Step 1: Searching legal corpus...
   Found 7 relevant chunks:
   [1] NBA_ABS_Regulations_2014.pdf (similarity: 0.77)
   [2] biological_diversity_act.pdf (similarity: 0.769)
   [3] NBA_ABS_Regulations_2014.pdf (similarity: 0.769)
   [4] biological_diversity_act.pdf (similarity: 0.767)
   [5] biological_diversity_act.pdf (similarity: 0.755)
   [6] biological_diversity_act.pdf (similarity: 0.755)
   [7] biological_diversity_act.pdf (similarity: 0.754)

🤖 Step 2: Generating cited answer via openai/gpt-oss-120b...
   Generated in 3.7 seconds

🏛️  IP-SAKTI SAHAYAK — RESPONSE
**Answer**  
As a foreign researcher you must obtain **prior approval (an ABS permit) from the National Biodiversity Authority (NBA)** before collecting or accessing any medicinal plant or associated tradition

## Test 3: Drug Classification

Can it classify an Ayurvedic product correctly?

In [12]:
answer = ask_ip_sakti(
    query="I want to manufacture and sell Chyawanprash. Is it classified as a classical "
          "Ayurvedic medicine, a proprietary medicine, or a food product? "
          "What license do I need?",
)

🔍 QUESTION: I want to manufacture and sell Chyawanprash. Is it classified as a classical Ayurvedic medicine, a proprietary medicine, or a food product? What license do I need?

📚 Step 1: Searching legal corpus...
   Found 7 relevant chunks:
   [1] Drugs_Rules_1945_Complete.pdf (similarity: 0.775)
   [2] First_Schedule_Authoritative_Books_ASU.pdf (similarity: 0.775)
   [3] Drugs_Cosmetics_Rules_Part_XVII_Labelling_ASU.pdf (similarity: 0.775)
   [4] Drugs_Cosmetics_Rules_Part_XVI_ASU.pdf (similarity: 0.775)
   [5] Schedule_T_GMP_for_ASU_Drugs.pdf (similarity: 0.775)
   [6] Drugs_Cosmetics_Rules_Part_XVII_Labelling_ASU.pdf (similarity: 0.77)
   [7] Drugs_Cosmetics_Rules_Part_XVI_ASU.pdf (similarity: 0.77)

🤖 Step 2: Generating cited answer via openai/gpt-oss-120b...
   Generated in 3.0 seconds

🏛️  IP-SAKTI SAHAYAK — RESPONSE
**Answer**  
Based on the sources provided, the only licensing framework mentioned for products such as Chyawanprash is the **“Licence to manufacture for sale of Ayu

## Test 4: International Treaty

Can it handle cross-jurisdiction questions?

In [13]:
answer = ask_ip_sakti(
    query="How does the WIPO GRATK Treaty 2024 protect traditional knowledge holders "
          "from biopiracy? What are the disclosure requirements for patent applicants?",
    jurisdiction="international",
)

🔍 QUESTION: How does the WIPO GRATK Treaty 2024 protect traditional knowledge holders from biopiracy? What are the disclosure requirements for patent applicants?
   Filters: jurisdiction=international

📚 Step 1: Searching legal corpus...
   Found 7 relevant chunks:
   [1] WIPO_GRATK_Treaty_2024_Final_Text.txt (similarity: 0.833)
   [2] WIPO_GRATK_Treaty_2024_Final_Text.pdf (similarity: 0.831)
   [3] WIPO_GRATK_Treaty_2024.pdf (similarity: 0.831)
   [4] WIPO_GRATK_Treaty_2024_Final_Text.txt | Article 3 (similarity: 0.745)
   [5] WIPO_GRATK_Treaty_2024_Final_Text.pdf | ARTICLE 4 (similarity: 0.744)
   [6] WIPO_GRATK_Treaty_2024.pdf | ARTICLE 4 (similarity: 0.744)
   [7] WIPO_GRATK_Treaty_2024_Final_Text.txt | ARTICLE 4 (similarity: 0.743)

🤖 Step 2: Generating cited answer via openai/gpt-oss-120b...
   Generated in 4.2 seconds

🏛️  IP-SAKTI SAHAYAK — RESPONSE
**Answer**  
The WIPO Treaty on Intellectual Property, Genetic Resources and Associated Traditional Knowledge (GRATK) 2024 seeks t

## Test 5: Biopiracy Case Study

Does the system know about the famous Turmeric and Neem cases?

In [14]:
answer = ask_ip_sakti(
    query="Tell me about the Turmeric patent case. Who filed it, why was it revoked, "
          "and what lessons does it teach about protecting traditional knowledge?",
    domain="traditional_knowledge",
)

🔍 QUESTION: Tell me about the Turmeric patent case. Who filed it, why was it revoked, and what lessons does it teach about protecting traditional knowledge?
   Filters: domain=traditional_knowledge

📚 Step 1: Searching legal corpus...
   Found 7 relevant chunks:
   [1] Turmeric_Patent_Revocation_Case_Study_USPTO_5401504.html (similarity: 0.733)
   [2] Turmeric_Patent_Revocation_Case_Study_USPTO_5401504.txt (similarity: 0.733)
   [3] Turmeric_Patent_Revocation_Case_Study_USPTO_5401504.html (similarity: 0.727)
   [4] Turmeric_Patent_Revocation_Case_Study_USPTO_5401504.txt (similarity: 0.727)
   [5] Turmeric_Patent_Revocation_Case_Study_USPTO_5401504.html (similarity: 0.704)
   [6] Turmeric_Patent_Revocation_Case_Study_USPTO_5401504.txt (similarity: 0.704)
   [7] Neem_Patent_Revocation_Case_Study_EPO_0436257.html (similarity: 0.703)

🤖 Step 2: Generating cited answer via openai/gpt-oss-120b...
   Generated in 2.5 seconds

🏛️  IP-SAKTI SAHAYAK — RESPONSE
### Answer
The case concerns U.S. P

## Response Quality Check

evaluate the quality of the last response.  
This checks if the AI is actually citing sources and not hallucinating.

In [15]:
import re

def evaluate_response(answer: str) -> dict:
    """
    Quick quality check on the AI's response.
    Checks for: citations, disclaimer, structure, length.
    """
    checks = {}
    
    # Check 1: Are there inline citations?
    citations = re.findall(r'\[Source\s*\d+\]', answer)
    checks["citations_found"] = len(citations)
    checks["has_citations"] = len(citations) > 0
    
    # Check 2: Is there a disclaimer?
    checks["has_disclaimer"] = any(word in answer.lower() 
                                    for word in ["disclaimer", "not legal advice",
                                                 "consult", "qualified"])
    
    # Check 3: Does it have structure (headers)?
    checks["has_structure"] = answer.count("###") >= 2 or answer.count("**") >= 4
    
    # Check 4: Response length
    checks["word_count"] = len(answer.split())
    checks["adequate_length"] = checks["word_count"] > 100
    
    # Check 5: Does it mention "I could not find" (honest about gaps)?
    checks["admits_gaps"] = "could not find" in answer.lower() or \
                            "not enough information" in answer.lower()
    
    # Overall score
    score = 0
    if checks["has_citations"]: score += 30
    if checks["citations_found"] >= 3: score += 10
    if checks["has_disclaimer"]: score += 20
    if checks["has_structure"]: score += 20
    if checks["adequate_length"]: score += 20
    checks["quality_score"] = score
    
    # Display
    print(f"📊 RESPONSE QUALITY CHECK")
    print(f"{'─'*40}")
    print(f"  Citations found  : {checks['citations_found']} {'✅' if checks['has_citations'] else '❌'}")
    print(f"  Has disclaimer   : {'✅' if checks['has_disclaimer'] else '❌'}")
    print(f"  Structured format: {'✅' if checks['has_structure'] else '❌'}")
    print(f"  Word count       : {checks['word_count']} {'✅' if checks['adequate_length'] else '❌'}")
    print(f"  Admits gaps      : {'✅' if checks['admits_gaps'] else '➖ (not needed)'}")
    print(f"{'─'*40}")
    grade = '🟢 EXCELLENT' if score >= 80 else '🟡 GOOD' if score >= 60 else '🔴 NEEDS IMPROVEMENT'
    print(f"  Quality Score    : {score}/100 {grade}")
    
    return checks


# Evaluate the last answer
if 'answer' in dir():
    evaluate_response(answer)
else:
    print("Run one of the test cells (10-14) first, then run this cell.")

📊 RESPONSE QUALITY CHECK
────────────────────────────────────────
  Citations found  : 3 ✅
  Has disclaimer   : ✅
  Structured format: ✅
  Word count       : 386 ✅
  Admits gaps      : ➖ (not needed)
────────────────────────────────────────
  Quality Score    : 100/100 🟢 EXCELLENT


---

 **fully working RAG pipeline**:

```
User Question (natural language)
        ↓
BGE-Small Embedding (converts to vector)
        ↓
ChromaDB Search (finds top 7 legal chunks)
        ↓
Source Formatting (numbered [Source 1], [Source 2]...)
        ↓
Groq API → Llama 3.3 70B (generates cited answer)
        ↓
Structured Response with Citations + Disclaimer
```
